In [24]:
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Dataset

In [25]:
class KeypointDataset(Dataset):
    def __init__(self, csv_path):
        self.samples = []
        with open(csv_path, "r") as f:
            reader = csv.reader(f)
            for row in reader:
                label = int(row[0])
                points = list(map(float, row[1:]))
                self.samples.append((points, label))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        points, label = self.samples[idx]
        points = torch.tensor(points, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)
        return points, label


# Model

In [26]:
# Initialize the model
from keypoint_classifier import KeyPointModel

# Train

In [29]:
# Paths
dataset_path = "dataset/keypoint.csv"
dataset = KeypointDataset(dataset_path)
NUM_CLASSES = len(set([label for _, label in dataset.samples]))

train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = KeyPointModel(NUM_CLASSES).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100

for epoch in range(EPOCHS):
	total_loss = 0
	
	for points, labels in train_loader:
		points, labels = points.to(device), labels.to(device)
		
		optimizer.zero_grad()
		
		logits = model(points)
		loss = nn.functional.cross_entropy(logits, labels)
		
		loss.backward()
		optimizer.step()
		
		total_loss += loss.item()
	
	cur_loss = total_loss / len(train_loader)
	print(f"Epoch {epoch+1}, loss: {cur_loss}")

model_save_path = "keypoint_model.pth"
torch.save(model.state_dict(), model_save_path)

Epoch 1, loss: 1.1261468331019084
Epoch 2, loss: 1.1216264565785725
Epoch 3, loss: 1.1177965799967449
Epoch 4, loss: 1.108924150466919
Epoch 5, loss: 1.1094352006912231
Epoch 6, loss: 1.0991955995559692
Epoch 7, loss: 1.0949249267578125
Epoch 8, loss: 1.0999445120493572
Epoch 9, loss: 1.097756028175354
Epoch 10, loss: 1.084277629852295
Epoch 11, loss: 1.0915309190750122
Epoch 12, loss: 1.084899107615153
Epoch 13, loss: 1.0743779341379802
Epoch 14, loss: 1.0835016965866089
Epoch 15, loss: 1.0662637154261272
Epoch 16, loss: 1.0623782873153687
Epoch 17, loss: 1.0408504009246826
Epoch 18, loss: 1.0491649309794109
Epoch 19, loss: 1.0425291061401367
Epoch 20, loss: 1.0268736084302266
Epoch 21, loss: 1.0282432635625203
Epoch 22, loss: 1.0285062392552693
Epoch 23, loss: 1.026527225971222
Epoch 24, loss: 1.0244997541109722
Epoch 25, loss: 1.0217799345652263
Epoch 26, loss: 1.0084345539410908
Epoch 27, loss: 1.0115352670351665
Epoch 28, loss: 0.9991132815678915
Epoch 29, loss: 0.9885889490445455